In [1]:
import pandas as pd
import numpy as np

import os
import json
import joblib
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from catboost import CatBoostClassifier

In [ ]:
monitoring_report = pd.read_csv(
    r"data\processed\week3_monitoring_report.csv"
)

monitoring_report

,batch_id,max_numerical_psi,numerical_drift_features,categorical_drift_features,drift_status,accuracy,precision,recall,f1,roc_auc,f1_degradation_pct,performance_status,reliability_status,retraining_signal
0,batch_001,0.075798,0,0,HEALTHY,0.800712,0.644737,0.628205,0.636364,0.864090,-1.010101,Healthy,HEALTHY,NO ACTION
1,batch_002,0.022588,0,2,HEALTHY,0.782918,0.573770,0.500000,0.534351,0.811510,15.182358,Degraded,PERFORMANCE DEGRADATION,RETRAIN
2,batch_003,0.085334,0,1,HEALTHY,0.790036,0.620253,0.628205,0.624204,0.849312,0.920028,Healthy,HEALTHY,NO ACTION
3,batch_004,0.031496,0,0,HEALTHY,0.775801,0.530864,0.632353,0.577181,0.838822,8.383935,Healthy,HEALTHY,NO ACTION
4,batch_005,0.039982,0,1,HEALTHY,0.824561,0.702703,0.650000,0.675325,0.909878,-7.194393,Healthy,HEALTHY,NO ACTION


In [3]:
retraining_required = monitoring_report[
    monitoring_report["retraining_signal"] == "RETRAIN"
]

retraining_required

,batch_id,max_numerical_psi,numerical_drift_features,categorical_drift_features,drift_status,accuracy,precision,recall,f1,roc_auc,f1_degradation_pct,performance_status,reliability_status,retraining_signal
1,batch_002,0.022588,0,2,HEALTHY,0.782918,0.57377,0.5,0.534351,0.81151,15.182358,Degraded,PERFORMANCE DEGRADATION,RETRAIN


In [4]:
if len(retraining_required) > 0:
    print("⚠️ Retraining is required.")
else:
    print("✅ No retraining required.")

⚠️ Retraining is required.


In [ ]:
reference_data = pd.read_csv(
    r"data\processed\reference_data.csv"
)

In [ ]:
production_batches = []

production_labels = []

for i in range(1, 6):

    X_batch = pd.read_csv(
        fr"data\production\batch_{i:03d}.csv"
    )

    y_batch = pd.read_csv(
        fr"data\production\labels_{i:03d}.csv"
    )

    production_batches.append(X_batch)

    production_labels.append(
        y_batch.iloc[:, 0]
    )

In [7]:
X_production = pd.concat(
    production_batches,
    ignore_index=True
)

y_production = pd.concat(
    production_labels,
    ignore_index=True
)

print("Production X:", X_production.shape)
print("Production y:", y_production.shape)

Production X: (1409, 19)
Production y: (1409,)


In [8]:
X_retrain = pd.concat(
    [
        reference_data,
        X_production
    ],
    ignore_index=True
)

print(
    "Retraining features:",
    X_retrain.shape
)

Retraining features: (5916, 19)


In [ ]:
reference_labels = pd.read_csv(
    r"data\processed\reference_labels.csv"
)

reference_labels = reference_labels.iloc[:, 0]

In [10]:
y_retrain = pd.concat(
    [
        reference_labels,
        y_production
    ],
    ignore_index=True
)

print(
    "Retraining target:",
    y_retrain.shape
)

Retraining target: (5916,)


In [11]:
print(X_retrain.shape)
print(y_retrain.shape)

(5916, 19)
(5916,)


In [12]:
print(
    y_retrain.value_counts()
)

print(
    y_retrain.value_counts(normalize=True)
)

Churn Value
0    4346
1    1570
Name: count, dtype: int64
Churn Value
0    0.734618
1    0.265382
Name: proportion, dtype: float64


In [13]:
categorical = [
    'Multiple Lines',
    'Internet Service',
    'Online Security',
    'Online Backup',
    'Device Protection',
    'Tech Support',
    'Streaming TV',
    'Streaming Movies',
    'Contract',
    'Payment Method'
]

num_cols = [
    'Monthly Charges',
    'Total Charges',
    'Tenure Months'
]

In [14]:
binary_columns = [
    'Gender',
    'Senior Citizen',
    'Partner',
    'Dependents',
    'Phone Service',
    'Paperless Billing'
]

for col in binary_columns:

    print(
        col,
        reference_data[col].unique()
    )

Gender [1 0]
Senior Citizen [0 1]
Partner [0 1]
Dependents [0 1]
Phone Service [1 0]
Paperless Billing [0 1]


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore" ,
                drop='first'
            ),
            categorical
        )
    ] ,
    remainder='passthrough'
)

In [16]:
X_train_v2, X_val_v2, y_train_v2, y_val_v2 = train_test_split(
    X_retrain,
    y_retrain,
    test_size=0.20,
    random_state=42,
    stratify=y_retrain
)

In [17]:
print(
    "Training:",
    X_train_v2.shape
)

print(
    "Validation:",
    X_val_v2.shape
)

Training: (4732, 19)
Validation: (1184, 19)


In [18]:
pipeline_v2 = Pipeline([
    
    (
        "preprocessor",
        preprocessor
    ),
    
    (
        "smote",
        SMOTE(
            sampling_strategy=0.6,
            random_state=42
        )
    ),
    
    (
        "cat",
        CatBoostClassifier(
            verbose=0,
            random_state=42,
            max_depth=5,
            learning_rate=0.03,
            n_estimators=300
        )
    )
])

In [19]:
print("Training Model v2...")

pipeline_v2.fit(
    X_train_v2,
    y_train_v2
)

print(
    "✅ Model v2 training completed."
)

Training Model v2...
✅ Model v2 training completed.


In [20]:
pred_v2 = pipeline_v2.predict(
    X_val_v2
)

prob_v2 = pipeline_v2.predict_proba(
    X_val_v2
)[:, 1]

In [21]:
accuracy_v2 = accuracy_score(
    y_val_v2,
    pred_v2
)

precision_v2 = precision_score(
    y_val_v2,
    pred_v2
)

recall_v2 = recall_score(
    y_val_v2,
    pred_v2
)

f1_v2 = f1_score(
    y_val_v2,
    pred_v2
)

roc_auc_v2 = roc_auc_score(
    y_val_v2,
    prob_v2
)

In [22]:
print("================================")
print("MODEL V2 PERFORMANCE")
print("================================")

print(
    f"Accuracy : {accuracy_v2:.4f}"
)

print(
    f"Precision: {precision_v2:.4f}"
)

print(
    f"Recall   : {recall_v2:.4f}"
)

print(
    f"F1 Score : {f1_v2:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_v2:.4f}"
)

MODEL V2 PERFORMANCE
Accuracy : 0.8125
Precision: 0.6474
Recall   : 0.6433
F1 Score : 0.6454
ROC-AUC  : 0.8595


In [23]:
baseline_f1 = 0.6363636363636364
baseline_accuracy = 0.800711743772242
baseline_precision = 0.6447368421052632
baseline_recall = 0.6282051282051282
baseline_roc_auc = 0.8640899330554502

In [24]:
comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "Model_v1": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_roc_auc
    ],

    "Model_v2": [
        accuracy_v2,
        precision_v2,
        recall_v2,
        f1_v2,
        roc_auc_v2
    ]
})

comparison

,Metric,Model_v1,Model_v2
0,Accuracy,0.800712,0.812500
1,Precision,0.644737,0.647436
2,Recall,0.628205,0.643312
3,F1,0.636364,0.645367
4,ROC-AUC,0.864090,0.859453


In [25]:
f1_improvement = (
    f1_v2 -
    baseline_f1
)

print(
    f"F1 improvement: {f1_improvement:.4f}"
)

F1 improvement: 0.0090


In [26]:
f1_improvement_pct = (
    f1_improvement /
    baseline_f1
) * 100

print(
    f"F1 improvement percentage: "
    f"{f1_improvement_pct:.2f}%"
)

F1 improvement percentage: 1.41%


In [27]:
minimum_improvement = 0.01
if f1_improvement >= minimum_improvement:

    promote_v2 = True

else:

    promote_v2 = False

In [28]:
if promote_v2:

    print(
        "🚀 Model v2 PASSED validation."
    )

else:

    print(
        "🛑 Model v2 FAILED validation."
    )

🛑 Model v2 FAILED validation.


In [ ]:
model_v2_path = (
    r"models\v2\model.pkl"
)

joblib.dump(
    pipeline_v2,
    model_v2_path
)

print(
    "Model v2 saved to:",
    model_v2_path
)

Model v2 saved to: C:\Users\ah266\OneDrive\Documents\production-ml-reliability\models\v2\model.pkl


In [30]:
model_v2_metadata = {

    "model_version": "v2",

    "model_type":
        "CatBoostClassifier",

    "accuracy":
        float(accuracy_v2),

    "precision":
        float(precision_v2),

    "recall":
        float(recall_v2),

    "f1":
        float(f1_v2),

    "roc_auc":
        float(roc_auc_v2),

    "parent_model":
        "v1",

    "training_data":
        "reference + production",

    "training_rows":
        int(len(X_train_v2)),

    "validation_rows":
        int(len(X_val_v2)),

    "trained_at":
        datetime.now().isoformat()
}

In [ ]:
with open(
    r"models\v2\metadata.json",
    "w"
) as f:

    json.dump(
        model_v2_metadata,
        f,
        indent=4
    )

In [32]:
registry = {

    "active_model": "v1",

    "models": {

        "v1": {
            "status": "production",
            "f1": float(baseline_f1)
        },

        "v2": {
            "status": "candidate",
            "f1": float(f1_v2)
        }
    }
}

In [33]:
if promote_v2:

    registry["active_model"] = "v2"

    registry["models"]["v1"]["status"] = "archived"

    registry["models"]["v2"]["status"] = "production"

    active_model = "v2"

    print(
        "🚀 Model v2 promoted to production."
    )

else:

    registry["active_model"] = "v1"

    registry["models"]["v1"]["status"] = "production"

    registry["models"]["v2"]["status"] = "rejected"

    active_model = "v1"

    print(
        "🛑 Model v2 rejected."
    )

    print(
        "Model v1 remains active."
    )

🛑 Model v2 rejected.
Model v1 remains active.


In [ ]:
with open(
    r"models\model_registry.json",
    "w"
) as f:

    json.dump(
        registry,
        f,
        indent=4
    )

In [ ]:
with open(
    r"models\model_registry.json",
    "r"
) as f:

    registry_loaded = json.load(f)

active_model = registry_loaded[
    "active_model"
]

print(
    "Active model:",
    active_model
)

Active model: v1


In [ ]:
active_model_path = (
    fr"models\{active_model}\model.pkl"
)

active_pipeline = joblib.load(
    active_model_path
)

print(
    f"✅ Loaded active model: {active_model}"
)

✅ Loaded active model: v1


In [38]:
test_batch = production_batches[0]

test_predictions = active_pipeline.predict(
    test_batch
)

print(
    "Predictions generated:",
    len(test_predictions)
)

Predictions generated: 281


In [ ]:
def rollback_to_v1():

    registry["active_model"] = "v1"

    registry["models"]["v1"]["status"] = "production"

    registry["models"]["v2"]["status"] = "archived"

    with open(
        r"models\model_registry.json",
        "w"
    ) as f:

        json.dump(
            registry,
            f,
            indent=4
        )

    print(
        "⚠️ Rollback completed."
    )

    print(
        "Active model: v1"
    )

In [ ]:
#rollback_to_v1()

In [40]:
retraining_report = {

    "timestamp":
        datetime.now().isoformat(),

    "previous_model":
        "v1",

    "candidate_model":
        "v2",

    "previous_f1":
        float(baseline_f1),

    "new_f1":
        float(f1_v2),

    "f1_improvement":
        float(f1_improvement),

    "f1_improvement_percentage":
        float(f1_improvement_pct),

    "minimum_required_improvement":
        float(minimum_improvement),

    "model_promoted":
        bool(promote_v2),

    "active_model":
        active_model
}

In [ ]:
with open(
    r"data\processed\retraining_report.json",
    "w"
) as f:

    json.dump(
        retraining_report,
        f,
        indent=4
    )

In [42]:
print("======================================")
print("AUTONOMOUS RETRAINING RESULT")
print("======================================")

print(
    f"Model v1 F1: {baseline_f1:.4f}"
)

print(
    f"Model v2 F1: {f1_v2:.4f}"
)

print(
    f"Improvement: {f1_improvement:.4f}"
)

print(
    f"Required Improvement: "
    f"{minimum_improvement:.4f}"
)

print(
    f"Model v2 promoted: "
    f"{promote_v2}"
)

print(
    f"Active model: "
    f"{active_model}"
)

AUTONOMOUS RETRAINING RESULT
Model v1 F1: 0.6364
Model v2 F1: 0.6454
Improvement: 0.0090
Required Improvement: 0.0100
Model v2 promoted: False
Active model: v1


In [43]:
print("\n========== FINAL MODEL STATUS ==========")

print(
    "Active Model:",
    active_model
)

print(
    "Model v1 F1:",
    round(baseline_f1, 4)
)

print(
    "Model v2 F1:",
    round(f1_v2, 4)
)

print(
    "Improvement:",
    round(f1_improvement, 4)
)

print(
    "Promotion:",
    "YES" if promote_v2 else "NO"
)


========== FINAL MODEL STATUS ==========
Active Model: v1
Model v1 F1: 0.6364
Model v2 F1: 0.6454
Improvement: 0.009
Promotion: NO
